# SoDA: from a notebook to a working app

Train → Package → Deploy → Monitor

We use public handwritten digits to keep the model small and the deployment understandable. Run these cells in order with the Python 3.12 kernel.

In [ ]:
# Setup: imports give us data loading, model training, evaluation, and saving tools.
import sys
from pathlib import Path

# Jupyter may start in the repo root or notebooks/. Find the same artifacts folder.
ROOT = Path.cwd()
if not (ROOT / "ml").exists():
    ROOT = ROOT.parent
assert (ROOT / "ml/train.py").exists(), "Open this notebook from the workshop repository"
sys.path.insert(0, str(ROOT))
import json

import joblib
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

from ml.train import split_data, train

## 1. Inspect the data

Each image has 8 × 8 grayscale pixels. Zero is black and 16 is bright. The target is the digit 0–9. This dataset ships with scikit-learn.

Source: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html

In [ ]:
# This small dataset is bundled with scikit-learn; no download or GPU is needed.
digits = load_digits()
# data: 1,797 rows of 64 input features (8x8 flattened row by row).
# target: the known digit for each row. Labels teach/evaluate; they are NOT inputs.
print("Examples:", len(digits.data))
print("Features:", digits.data.shape[1])
print("First label:", digits.target[0])
# images is the same data reshaped to 8x8: 0 = black, 16 = white.
print(digits.images[0].astype(int))

## 2. Split before fitting preprocessing

The fixed seed gives everyone the same split. Stratification preserves class proportions. The test set stays separate while we fit the scaler and model.

In [ ]:
# x = image features; y = correct answers. Reserve 20% for evaluation.
# Seed 42 fixes the split; stratification keeps digit proportions similar.
x_train, x_test, y_train, y_test = split_data()
print(f"Train: {len(x_train)}; test: {len(x_test)}")
# A pipeline keeps preprocessing and prediction together during training AND serving.
# MinMaxScaler learns per-feature ranges only from training data (avoid data leakage).
# SVC learns nonlinear class boundaries. C controls the penalty for training errors.
# probability=True provides class scores; a high score can still be wrong.
pipeline = make_pipeline(
    MinMaxScaler(), SVC(C=10.0, probability=True, random_state=42)
)
# fit learns from examples. Never pass x_test/y_test here or fit inside the API.
pipeline.fit(x_train, y_train)

## 3. Evaluate on held-out examples

The confusion matrix uses actual labels as rows and predicted labels as columns. A good score here does not guarantee success on your own handwriting.

**Discuss:** Why does the test set need to stay outside `fit`?

The browser crops, centers, and averages drawings into 8 × 8 pixels before calling the API. That image preparation matters: shrinking the entire canvas made small drawings look unlike the training data. The nonlinear SVC handles these shapes better than our original linear baseline. Its highest-probability class can differ from `SVC.predict()`, so evaluation and serving both use `predict_proba().argmax()`.


In [ ]:
# Match the API: choose the class with the highest probability.
# Each output row contains 10 scores. axis=1 picks the biggest score in each row.
# classes_ maps score positions to digit labels; no training happens in this call.
probabilities = pipeline.predict_proba(x_test)
predictions = pipeline.classes_[probabilities.argmax(axis=1)]
print(f"Held-out accuracy: {accuracy_score(y_test, predictions):.2%}")
# Accuracy = correct / total. Matrix rows are true digits; columns are predictions.
# Diagonal entries are correct; off-diagonal entries show which digits get confused.
# This reused split is a regression benchmark, not proof about new handwriting.
print(confusion_matrix(y_test, predictions))

## 4. Package the complete pipeline

The reusable training command applies a 93% quality gate and saves the fitted preprocessing with the classifier. It also records the model checksum, split, evaluation, dependency versions, and input contract.

Only load joblib artifacts you created or trust. Loading one can execute Python code.

In [ ]:
# Use the reusable training function so notebook, CLI, and Docker build agree.
# It repeats fitting/evaluation, checks accuracy >= 93%, and only then saves.
# model.joblib = fitted scaler + classifier; samples.json = labeled demo inputs.
# metadata.json records the contract, score, versions, and artifact checksum.
# A checksum identifies bytes; it does not make an untrusted joblib file safe.
metadata = train(ROOT / "artifacts")
print(
    json.dumps(
        {
            k: metadata[k]
            for k in ["model_version", "test_accuracy", "train_samples", "test_samples", "input"]
        },
        indent=2,
    )
)

## 5. Reload and predict

This is the same packaged pipeline the API loads on startup. Restart the API after replacing its artifact.

In [ ]:
# Simulate service startup: load our trusted saved pipeline, without fitting again.
loaded = joblib.load(ROOT / "artifacts/model.joblib")
# Parity check: saving/reloading must preserve the predictions we just evaluated.
assert (loaded.classes_[loaded.predict_proba(x_test).argmax(axis=1)] == predictions).all()
# [0:1] keeps a 2-D batch shape (1, 64); the model expects rows of examples.
example = x_test[0:1]
print("Actual digit:", y_test[0])
print("Prediction:", loaded.classes_[loaded.predict_proba(example).argmax(axis=1)][0])
print("Class scores:", loaded.predict_proba(example).round(3))
# This is the exact JSON shape the browser/scripts will POST to /predict.
# tolist() converts NumPy values into ordinary JSON-compatible Python values.
print("Request JSON:", json.dumps({"pixels": example[0].tolist()}))

## Your next step

In a terminal at the repository root, run:

```bash
bash scripts/start.sh
```

Open port 8000. Select a held-out example and click Predict. In a second terminal, run `python scripts/smoke.py`. Follow `docs/WORKSHOP.md` for validation, deployment, and monitoring exercises.

**Exercise 2:** Run `python scripts/traffic.py --mode invalid --count 1`. Compare the original/sent images and JSON. Pixel 0 is 99, outside 0–16. HTTP 422 means validation rejected the request before the model ran. Inspect `detail`, `model_ran`, and the shared `request_id` in the terminal and dashboard error log.

**Exercise 3 — data drift:** Run `python scripts/traffic.py --mode shifted --count 30`. Adding 8 (capped at 16) simulates changed camera exposure. The input distribution changes while the known digit stays the same. Compare original/brighter images and actual predictions in the dashboard. A wrong prediction can still return HTTP 200. A brightness alert alone does not measure accuracy; here we can check outcomes because our teaching samples have labels.

**Read the monitoring JSON:** click **Stats JSON** in the app, or append `/stats` to its forwarded Codespaces URL. In a second terminal: `curl -s http://localhost:8000/stats | python -m json.tool`. GET reads statistics; POST `/predict` sends an image for inference.

**Extension:** Try `C=0.1` as a candidate in a separate output directory. Before doing systematic tuning, add a validation split so you do not repeatedly optimize against the test set.